In [34]:
import pandas as pd
from pathlib import Path
from functools import reduce
import numpy as np

In [35]:
raw_dir = Path.home() / "Documents" / "diabetes-risk-project" / "data" / "raw"

In [36]:
# Load and store the data 
files={
    "demo": "P_DEMO.xpt",
    "body" : "P_BMX.xpt",
    "bp_exam" : "P_BPXO.xpt", 
    "hba1c" : "P_GHB.xpt",
    "glucose" : "P_GLU.xpt",
    "diabetes_q" : "P_DIQ.xpt",
    "bp_chol_q" : "P_BPQ.xpt",
    "physical_activity" : "P_PAQ.xpt",
    "smoking" : "P_SMQ.xpt",
    "alcohol" : "P_ALQ.xpt",
    "diet" : "P_DBQ.xpt"
}

data={}
for name, filename in files.items():
    file_path=raw_dir/filename
    data[name]=pd.read_sas(file_path, format="xport")
    print(name, data[name].shape)

demo (15560, 29)
body (14300, 22)
bp_exam (11656, 12)
hba1c (10409, 2)
glucose (5090, 4)
diabetes_q (14986, 28)
bp_chol_q (10195, 11)
physical_activity (9693, 17)
smoking (11137, 16)
alcohol (8965, 10)
diet (15560, 46)


In [37]:
for name, df in data.items():
    print(name, "SEQN" in df.columns)

demo True
body True
bp_exam True
hba1c True
glucose True
diabetes_q True
bp_chol_q True
physical_activity True
smoking True
alcohol True
diet True


In [38]:
# Merge all files 
dfs=list(data.values())
data_merged = reduce(lambda left, right: pd.merge(left, right, on="SEQN", how="outer"), dfs)
data_merged.shape
#data_merged.head()

(15560, 187)

In [39]:
# store the processed raw merged data
# store the processed raw merged data
processed_dir = Path.home() / "Documents" / "diabetes-risk-project" / "data" / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

data_merged.to_csv(processed_dir / "data_merged_raw.csv", index=False)


In [40]:
# Project dataset 
project_cols = [
    "SEQN",

    # Demographics
    "RIDAGEYR",     # age
    "RIAGENDR",     # sex
    "RIDRETH3",     # race/ethnicity
    "DMDEDUC2",     # education
    "INDFMPIR",     # income-to-poverty ratio

    # Body measurement
    "BMXBMI",       # BMI
    "BMXWAIST",     # waist circumference

    # Blood pressure readings
    "BPXOSY1", "BPXOSY2", "BPXOSY3",   # systolic BP readings
    "BPXODI1", "BPXODI2", "BPXODI3",   # diastolic BP readings

    # Target construction
    "DIQ010",       # doctor told diabetes
    "LBXGH",        # glycohemoglobin / HbA1c
    "LBXGLU",       # fasting glucose

    # Medical history
    "BPQ020",       # told high blood pressure
    "BPQ080",       # told high cholesterol

    # Smoking
    "SMQ020",       # smoked at least 100 cigarettes

    # Physical activity
    "PAQ650", "PAQ655", "PAD660",       # vigorous recreational activity
    "PAQ665", "PAQ670", "PAD675",       # moderate recreational activity
    "PAD680",                           # sedentary minutes

    # Alcohol behavior
    "ALQ111", "ALQ121", "ALQ130",

    # Diet behavior
    "DBQ700",       # overall diet health
    "DBD895",       # meals not home prepared 
    "DBD900",        # meals from fast food or pizza place 
    "DBD905",        # of ready-to-eat foods in past 30 days 
    "DBD910"        # of frozen meals/ pizza in past 30 days 
]
df_project = data_merged[project_cols].copy()
print(df_project.shape)
df_project.head()

(15560, 35)


,SEQN,RIDAGEYR,RIAGENDR,RIDRETH3,DMDEDUC2,INDFMPIR,BMXBMI,BMXWAIST,BPXOSY1,BPXOSY2,...,PAD675,PAD680,ALQ111,ALQ121,ALQ130,DBQ700,DBD895,DBD900,DBD905,DBD910
0,109263.0,2.0,1.0,6.0,NaN,4.66,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,5.397605e-79,5.397605e-79,5.000000e+00
1,109264.0,13.0,2.0,1.0,NaN,0.83,17.6,63.8,109.0,109.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,5.397605e-79,1.000000e+00,1.000000e+00
2,109265.0,2.0,1.0,3.0,NaN,3.06,15.0,41.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.000000e+00,5.397605e-79,2.000000e+00
3,109266.0,29.0,2.0,6.0,5.0,5.00,37.8,117.9,99.0,99.0,...,30.0,480.0,1.0,10.0,1.0,3.0,7.0,5.397605e-79,5.397605e-79,5.000000e+00
4,109267.0,21.0,2.0,2.0,4.0,5.00,NaN,NaN,NaN,NaN,...,NaN,540.0,NaN,NaN,NaN,1.0,4.0,5.397605e-79,5.397605e-79,5.397605e-79


In [41]:
df_project.to_csv(processed_dir/"project_dataset.csv", index=False)